# `the_one.ipynb` — optimized four-factor socle multiplicities

This notebook optimizes the **specific** coefficient implemented in the original `the_one.ipynb`, **not** the three-factor Hoyt formula. It requires a **SageMath kernel**, because it uses `sage.libs.lrcalc.lrcalc`.

For outer partitions $\lambda,\mu$ and residual partitions $\lambda',\mu'$, the original code computes

$$
T_k^{\lambda,\mu}(\lambda',\mu')=
\sum_{\substack{\delta,\gamma,\,p,q\ge0\\
|\delta|+2|\gamma|+p+q=k}}
N^{\lambda}_{\lambda',\gamma,\delta,(1^p)}
N^{\mu}_{\mu',\gamma,\delta,(1^q)}.
$$

Here $(1^p)$ is a **column of $p$ boxes**, with $(1^0)=\varnothing$. The factors $\delta,\gamma$ are shared, but $p,q$ can differ. This is **not** the old Hoyt sum, and in particular **$k$ may exceed $\min(|\lambda|,|\mu|)$**.

## Algebraic optimization

Let $A=|\lambda|-|\lambda'|$ and $B=|\mu|-|\mu'|$. The degree constraints imply

$$
A=|\delta|+|\gamma|+p,\qquad
B=|\delta|+|\gamma|+q,\qquad
k=|\delta|+2|\gamma|+p+q.
$$

Writing $d=|\delta|$ and $g=|\gamma|$, eliminate $p,q$:

$$
\boxed{d=A+B-k},\qquad
p=A-d-g,\qquad q=B-d-g.
$$

Thus `solveOne()` needs only a **single loop over $g$**, rather than nested degree searches.

To avoid evaluating a four-factor LR coefficient repeatedly, let $\xi\prec_p\lambda$ mean $\lambda/\xi$ is a vertical $p$-strip. Pieri's rule and associativity give

$$
N^\lambda_{\lambda',\gamma,\delta,(1^p)}
=\sum_{\nu\vdash d+g}c^\nu_{\delta,\gamma}
\underbrace{\sum_{\substack{\xi\subseteq\lambda\\\lambda/\xi\ \mathrm{vertical}\ p\mathrm{-strip}}}
 c^\xi_{\nu,\lambda'}}_{L_{\lambda,p}(\lambda',\nu)}.
$$

Define $R_{\mu,q}(\mu',\omega)$ similarly and a **split-degree kernel**

$$
\boxed{K_{d,g}(\nu,\omega)
=\sum_{\substack{\delta\vdash d\\\gamma\vdash g}}
 c^\nu_{\delta,\gamma}c^\omega_{\delta,\gamma}}.
$$

The optimized coefficient is the exact contraction

$$
\boxed{
T_k^{\lambda,\mu}(\lambda',\mu')
=\sum_{\substack{d,g,p,q\ge0\\d+2g+p+q=k}}
\sum_{\nu,\omega\vdash d+g}
L_{\lambda,p}(\lambda',\nu)\,K_{d,g}(\nu,\omega)\,
R_{\mu,q}(\mu',\omega).
}
$$

Unlike the Hoyt kernel $K_n$, **we must preserve the split $(d,g)$**, since $\gamma$ carries weight two in the layer index.


In [11]:
from collections import defaultdict
from functools import lru_cache
from pathlib import Path
import sage.libs.lrcalc.lrcalc as lrcalc

def part(p):
    p = tuple(int(x) for x in p)
    if any(x <= 0 for x in p) or any(a < b for a, b in zip(p, p[1:])):
        raise ValueError(f"Invalid partition: {p}")
    return p

def degree(p):
    return sum(p)

@lru_cache(maxsize=None)
def partitions_of(n, cap=None):
    n = int(n)
    if n < 0:
        return ()
    if n == 0:
        return ((),)
    top = n if cap is None else min(n, int(cap))
    return tuple((i,) + rest
                 for i in range(top, 0, -1)
                 for rest in partitions_of(n-i, i))

@lru_cache(maxsize=None)
def coproduct(p):
    """All *ordered* LR coproduct pairs (a,b) -> c^p_{a,b}."""
    p = part(p)
    return {(part(a), part(b)): int(c)
            for (a,b), c in lrcalc.coprod(p, all=1).items()}

@lru_cache(maxsize=None)
def lr(outer, a, b):
    outer, a, b = map(part, (outer, a, b))
    if degree(outer) != degree(a) + degree(b):
        return 0
    return int(lrcalc.lrcoef_unsafe(outer, a, b))

# Compatibility with the original, but no brute-force intermediate partition loops.
@lru_cache(maxsize=None)
def lrcoef4(a, b, c, d, outer):
    a, b, c, d, outer = map(part, (a, b, c, d, outer))
    if degree(a)+degree(b)+degree(c)+degree(d) != degree(outer):
        return 0
    # Simpler, robust fallback using only actual product support.
    prod1 = lrcalc.mult(a, b)
    prod2 = lrcalc.mult(c, d)
    return sum(int(v1)*int(v2)*lr(outer, part(nu), part(om))
               for nu, v1 in prod1.items()
               for om, v2 in prod2.items())

print("SageMath four-factor LR helpers ready")

SageMath four-factor LR helpers ready


## Closed-form degree solutions and sparse Pieri/coproduct vectors

In [12]:
def solveOne(k, k1, k2, k3, k4):
    """Exact degree solutions (same dictionary fields as original the_one.ipynb)."""
    k, k1, k2, k3, k4 = map(int, (k, k1, k2, k3, k4))
    A, B = k1-k2, k3-k4
    d = A+B-k
    if min(k, A, B) < 0 or d < 0 or d > min(A,B):
        return []
    result = []
    for g in range(min(A-d, B-d)+1):
        p, q = A-d-g, B-d-g
        assert d + 2*g + p + q == k
        result.append({"deg δ": d, "deg γ": g, "p": p, "q": q,
                       "deg λ'": k2, "deg μ'": k4})
    return result

@lru_cache(maxsize=None)
def vertical_predecessors(outer, p):
    """All xi such that outer/xi is a vertical p-strip (one box per row)."""
    outer, p = part(outer), int(p)
    if p < 0 or p > len(outer):
        return ()
    ans = []
    def visit(i, previous, todo, values):
        if todo < 0 or todo > len(outer)-i:
            return
        if i == len(outer):
            if todo == 0:
                ans.append(tuple(x for x in values if x))
            return
        row = outer[i]
        for removed in (0, 1):
            new = row - removed
            if new >= 0 and new <= previous:
                visit(i+1, new, todo-removed, values+(new,))
    visit(0, float('inf'), p, ())
    return tuple(ans)

@lru_cache(maxsize=None)
def coproduct_by_left_degree(outer, n):
    """Map right factor -> {left factor: coefficient}, |left|=n."""
    outer, n = part(outer), int(n)
    grouped = defaultdict(dict)
    for (nu, lp), coeff in coproduct(outer).items():
        if degree(nu) == n:
            grouped[lp][nu] = int(coeff)
    return dict(grouped)

@lru_cache(maxsize=None)
def side_vectors(outer, p, n):
    """Map residual partition -> {nu : sum_{xi vertical p} c^xi_{nu,residual}}."""
    outer, p, n = part(outer), int(p), int(n)
    if p < 0 or n < 0 or p+n > degree(outer):
        return {}
    grouped = defaultdict(lambda: defaultdict(int))
    for xi in vertical_predecessors(outer,p):
        for lp, vec in coproduct_by_left_degree(xi,n).items():
            for nu, mult in vec.items():
                grouped[lp][nu] += mult
    return {lp: dict(vec) for lp, vec in grouped.items()}

@lru_cache(maxsize=None)
def split_coproduct(outer, d):
    """Restrict ordered pairs (delta,gamma) to |delta|=d."""
    outer, d = part(outer), int(d)
    return {pair: int(c) for pair,c in coproduct(outer).items()
            if degree(pair[0]) == d}

@lru_cache(maxsize=None)
def split_kernel(d, g, nu, omega):
    """K_{d,g}(nu,omega). d and g cannot be summed together."""
    d, g, nu, omega = int(d), int(g), part(nu), part(omega)
    if degree(nu) != d+g or degree(omega) != d+g:
        return 0
    if d == 0 or g == 0:
        return int(nu == omega)
    # Swapping d and g gives the same kernel, by commutativity of Schur products.
    d = min(d,g)
    if omega < nu:
        nu,omega = omega,nu
    a, b = split_coproduct(nu, d), split_coproduct(omega, d)
    if len(a) > len(b):
        a,b=b,a
    return sum(c*b.get(pair,0) for pair,c in a.items())

print("Closed-form degrees, vertical-strip vectors, and split kernels ready")

Closed-form degrees, vertical-strip vectors, and split kernels ready


## Complete optimized layer calculation

In [13]:
@lru_cache(maxsize=None)
def row_data(k, lam, mu):
    """Sparse {(lambda_prime, mu_prime): multiplicity}, exactly the_one's T_k."""
    k, lam, mu = int(k), part(lam), part(mu)
    nl, nm = degree(lam), degree(mu)
    if k < 0 or k > nl+nm:
        return {}
    result = defaultdict(int)
    # k = d + 2g + p + q. n=d+g is the shared Schur-product degree.
    for d in range(min(k,nl,nm)+1):
        for g in range((k-d)//2+1):
            n = d+g
            if n > min(nl,nm):
                continue
            strip_total = k-d-2*g  # p + q
            p_min = max(0, strip_total-min(len(mu),nm-n))
            p_max = min(strip_total, len(lam),nl-n)
            for p in range(p_min,p_max+1):
                q = strip_total-p
                left = side_vectors(lam,p,n)
                right = side_vectors(mu,q,n)
                if not left or not right:
                    continue
                nus = {nu for vec in left.values() for nu in vec}
                oms = {om for vec in right.values() for om in vec}
                nonzero_K = {(nu,om): split_kernel(d,g,nu,om)
                             for nu in nus for om in oms}
                for lp, lvec in left.items():
                    for mp, rvec in right.items():
                        total = sum(ca*cb*nonzero_K[(nu,om)]
                                    for nu,ca in lvec.items()
                                    for om,cb in rvec.items())
                        if total:
                            result[(lp,mp)] += total
    return dict(result)

def CalcSoc(k, lam, lamP, mu, muP):
    """Original function signature, now a lookup in a cached complete row."""
    return int(row_data(k,part(lam),part(mu)).get((part(lamP),part(muP)),0))

def max_degree(lam,mu):
    """Highest nonzero layer; note upper bound |lam|+|mu|, not min degrees."""
    lam,mu=part(lam),part(mu)
    return next((k for k in range(degree(lam)+degree(mu),-1,-1)
                 if row_data(k,lam,mu)), 0)

def degree_rows(lam,mu,include_zeros=False):
    lam,mu=part(lam),part(mu)
    for k in range(degree(lam)+degree(mu)+1):
        data=row_data(k,lam,mu)
        if data or include_zeros:
            yield k,data

print("Exact sparse multiplicity evaluator ready")

Exact sparse multiplicity evaluator ready


## Table generation

The original `MakeTable_tex` stops at its first empty row. This notebook instead checks **every possible** $k\in[0,|\lambda|+|\mu|]$, so it never assumes layers are consecutive. `write_table` creates parent directories and defaults to overwriting files rather than accidentally appending stale results.


In [14]:

def partition_intersection(lam, mu):
    return tuple(
        x for x in (
            min(a, b)
            for a, b in zip_longest(lam, mu, fillvalue=0)
        ) if x > 0
    )

def latex_part(p):
    p=part(p)
    return r"\varnothing" if not p else "("+",".join(map(str,p))+")"

def direct_sum(k,lam,mu,zero="0"):
    entries=row_data(k,part(lam),part(mu))
    terms=[]
    for (lp,mp),mult in sorted(entries.items(),reverse=True):
        prefix="" if mult==1 else str(mult)+r"\,"
        terms.append(prefix+r"V^{"+latex_part(lp)+","+latex_part(mp)+"}")
    return r" \oplus ".join(terms) if terms else zero

def DirectSumMethod(k,lam,mu):
    """Original interface: return empty string for an empty layer."""
    return direct_sum(k,lam,mu,zero="")

def table_tex(lam,mu,include_zeros=False):
    lam,mu=part(lam),part(mu)
    rows=[]
    for k in range(degree(lam)+degree(mu),-1,-1):
        expression=direct_sum(k,lam,mu)
        if (k==0 or k > degree(lam) - degree(mu)):
            # HEREEE
            if (include_zeros or expression!="0"):
                rows.append(rf"$k={k}:\quad {expression}$ \\ \hline")
    title=rf"Table for $\lambda={latex_part(lam)}$, $\mu={latex_part(mu)}$"
    return "\n".join([
        r"\begin{table}[htbp]", r"\centering",
        rf"\caption{{{title}}}",
        r"\renewcommand{\arraystretch}{1.35}",
        r"\begin{tabular}{|>{\centering\arraybackslash}p{0.93\linewidth}|}",
        r"\hline", *rows, r"\end{tabular}", r"\end{table}", ""
    ])

def write_table(lam,mu,filename,append=False,include_zeros=False):
    path=Path(filename)
    path.parent.mkdir(parents=True,exist_ok=True)
    with path.open('a' if append else 'w',encoding='utf8') as f:
        f.write(table_tex(lam,mu,include_zeros=include_zeros))
        f.write("\n")
    return path

def MakeTable_tex(lam,mu,filename):
    """Compatibility wrapper: old method appended by default."""
    return write_table(lam,mu,filename,append=True)

def MakeTable(k,lam,mu):
    """Print the LaTeX table for one pair (k retained for API compatibility)."""
    print(table_tex(lam,mu))

def Master1(k,lam,lamP,mu,muP,ex_string=''):
    """Original convenience entry point."""
    print(CalcSoc(k,lam,lamP,mu,muP))

def ones_partition(x):
    x=int(x)
    if x<0: raise ValueError('x must be nonnegative')
    return (1,)*x

def generate_tables(max_lam_degree,max_mu_degree,filename,
                    include_zeros=False, exploit_symmetry=False):
    from itertools import product
    max_lam_degree,max_mu_degree=map(int,(max_lam_degree,max_mu_degree))
    path=Path(filename)
    path.parent.mkdir(parents=True,exist_ok=True)
    with path.open('w',encoding='utf8') as f:
        f.write("% Requires the LaTeX array package\n")
        for n in range(max_lam_degree+1):
            for m in range(max_mu_degree+1):
                if exploit_symmetry and n>m:
                    continue
                for lam in partitions_of(n):
                    for mu in partitions_of(m):
                        if (exploit_symmetry and n==m and lam>mu)  :
                            continue
                        f.write(table_tex(lam,mu,include_zeros=include_zeros))
                        f.write("\n%--------------------\n")
    return path

def generateTables(n,m,file="tex_tables/t_all.tex"):
    return generate_tables(n,m,file)

def le_generateTables(max_m,max_n,filename="tex_tables/t_all.tex"):
    return generate_tables(max_n,max_m,filename)

# Examples, uncomment to execute:
# print(solveOne(2, 1, 0, 1, 0))
# print(row_data(2, (1,), (1,)))
# print(table_tex((1,1), (2,)))
# print(write_table((1,1), (2,), "tex_tables/example.tex"))
print(generate_tables(7,9,"tex_tables/t_all.tex", exploit_symmetry=True))

tex_tables/t_all.tex


## Regression against the original defining formula

This checks the **four-factor** coefficient and the original `solveOne` degree enumeration independently of the sparse kernel. Small complete tables are compared entry by entry; the tests do **not** assume the Hoyt `k` bounds.

SageMath is required to execute this cell. The baseline is deliberately slow and should be limited to small degrees.


In [15]:
@lru_cache(maxsize=None)
def reference_four_lr(a,b,c,d,outer):
    a,b,c,d,outer=map(part,(a,b,c,d,outer))
    if degree(a)+degree(b)+degree(c)+degree(d)!=degree(outer):
        return 0
    total=0
    for nu in partitions_of(degree(a)+degree(b)):
        x=lr(nu,a,b)
        if not x: continue
        for om in partitions_of(degree(c)+degree(d)):
            y=lr(om,c,d)
            if y: total+=x*y*lr(outer,nu,om)
    return total

def reference_solveOne(k,k1,k2,k3,k4):
    """Intentionally literal degree loops, independent of the closed-form solver."""
    if k2>k1 or k4>k3 or k<0:
        return []
    out=[]
    for d in range(k+1):
        for g in range((k-d)//2+1):
            p=k1-k2-d-g
            q=k3-k4-d-g
            if min(p,q)<0:continue
            if d+2*g+p+q==k:
                out.append((d,g,p,q))
    return out

def reference_coefficient(k,lam,lp,mu,mp):
    lam,lp,mu,mp=map(part,(lam,lp,mu,mp))
    out=0
    for d,g,p,q in reference_solveOne(k,degree(lam),degree(lp),degree(mu),degree(mp)):
        colp=(1,)*p
        colq=(1,)*q
        for delta in partitions_of(d):
            for gamma in partitions_of(g):
                out += reference_four_lr(lp,gamma,delta,colp,lam)*\
                       reference_four_lr(mp,gamma,delta,colq,mu)
    return out

# Original and optimized solver enumeration must coincide.
for k1 in range(5):
    for k3 in range(5):
        for k2 in range(k1+1):
            for k4 in range(k3+1):
                for k in range(k1+k3+1):
                    got = [(x['deg δ'],x['deg γ'],x['p'],x['q'])
                           for x in solveOne(k,k1,k2,k3,k4)]
                    assert got==reference_solveOne(k,k1,k2,k3,k4), (k,k1,k2,k3,k4)

# Complete multiplicity regression for small source partitions.
for lam in [(),(1,),(2,),(1,1),(2,1)]:
    for mu in [(),(1,),(2,),(1,1)]:
        for k in range(degree(lam)+degree(mu)+1):
            result=row_data(k,lam,mu)
            for lp in (x for n in range(degree(lam)+1) for x in partitions_of(n)):
                for mp in (y for n in range(degree(mu)+1) for y in partitions_of(n)):
                    got=result.get((lp,mp),0)
                    expected=reference_coefficient(k,lam,lp,mu,mp)
                    assert got==expected,(k,lam,lp,mu,mp,got,expected)

assert CalcSoc(2,(1,),(),(1,),())==2
assert max_degree((1,),(1,))>=2
print("Small-degree four-factor regression passed")

Small-degree four-factor regression passed


## Practical notes

- The source `the_one.ipynb` is unchanged. The `solveOne` degree solutions are preserved algebraically.
- Vertical-strip predecessors use Pieri's rule, avoiding brute-force intermediate partitions.
- Sparse LR coproducts and the split kernel are memoized (`functools.lru_cache`).
- `CalcSoc` preserves the original call signature, now reading an entire cached layer instead of recomputing one entry at a time.
- Full layers are examined through $|\lambda|+|\mu|$, not the smaller Hoyt upper bound.
- `lrcalc.coprod(..., all=1)` includes **ordered** pairs; do not insert a factor of two manually.
- Results need validation under **SageMath 10.7**, which is not installed in the current export environment. The mathematical reorganization is exact; a separate pure-Python combinatorial test is also supplied.

When generating large batches, begin with low degrees and inspect `coproduct.cache_info()`, `side_vectors.cache_info()`, `split_kernel.cache_info()`, and `row_data.cache_info()` to understand cache reuse.

In [16]:
# Only the two extremal nonzero layers: k_max and k=0.
# Requires the existing functions part, degree, latex_part, row_data,
# max_degree, and partitions_of from the optimized notebook.
from pathlib import Path


def _extremal_terms(k, lam, mu):
    """LaTeX terms for the actual sparse layer, without recomputation."""
    data = row_data(int(k), part(lam), part(mu))
    terms = []
    for (lp, mp), multiplicity in sorted(data.items(), reverse=True):
        factor = '' if int(multiplicity) == 1 else str(int(multiplicity)) + r'\,'
        terms.append(factor + r'V^{' + latex_part(lp) + ',' + latex_part(mp) + '}')
    return terms or ['0']


def _wrap_extremal_terms(terms, max_terms_per_line=2, max_chars=68):
    """Split long ⊕ sums at term boundaries; do not truncate any terms."""
    if max_terms_per_line < 1 or max_chars < 1:
        raise ValueError('max_terms_per_line and max_chars must be positive')
    chunks, current = [], []
    for term in terms:
        candidate = current + [term]
        text_length = len(r' \oplus '.join(candidate))
        if current and (len(candidate) > max_terms_per_line or text_length > max_chars):
            chunks.append(current)
            current = [term]
        else:
            current = candidate
    if current:
        chunks.append(current)
    return chunks


def table_top_bottom_tex(lam, mu, max_terms_per_line=2, max_chars=68):
    """LaTeX longtable with only k=k_max and k=0 (no duplicate if k_max=0).

    Long direct sums are split over continuation lines. A longtable can
    break between these lines at page boundaries.
    """
    lam, mu = part(lam), part(mu)
    top = int(max_degree(lam, mu))
    selected_layers = [top] if top == 0 else [top, 0]
    title = (r'Extremal layers for $\lambda=' + latex_part(lam)
             + r'$, $\mu=' + latex_part(mu) + r'$')
    result = [
        r'\begingroup', r'\small',
        r'\setlength{\LTpre}{6pt}', r'\setlength{\LTpost}{6pt}',
        r'\begin{longtable}{@{}p{0.14\linewidth}@{\hspace{0.8em}}p{0.79\linewidth}@{}}',
        r'\caption{' + title + r'}\\',
        r'\toprule', r'Layer & Decomposition \\', r'\midrule',
        r'\endfirsthead',
        r'\toprule', r'Layer & Decomposition (continued) \\', r'\midrule',
        r'\endhead',
    ]
    for layer_index, k in enumerate(selected_layers):
        chunks = _wrap_extremal_terms(
            _extremal_terms(k, lam, mu),
            max_terms_per_line=max_terms_per_line,
            max_chars=max_chars,
        )
        for i, chunk in enumerate(chunks):
            key = rf'$k={k}$' if i == 0 else ''
            expression = r' \oplus '.join(chunk)
            if i > 0:
                expression = r'{}\oplus ' + expression
            result.append(key + r' & $\displaystyle ' + expression + r'$ \\')
        if layer_index < len(selected_layers) - 1:
            result.append(r'\addlinespace[5pt]\midrule\addlinespace[5pt]')
    result += [r'\bottomrule', r'\end{longtable}', r'\endgroup', '']
    return '\n'.join(result)


def _top_bottom_document_start(title='Extremal socle layers', author=''):
    def escape_latex(s):
        return (str(s).replace('\\', r'\textbackslash{}')
                .replace('&', r'\&').replace('%', r'\%')
                .replace('_', r'\_').replace('#', r'\#'))
    return '\n'.join([
        r'\documentclass[a4paper,11pt]{article}',
        r'\usepackage[margin=22mm]{geometry}',
        r'\usepackage[T1]{fontenc}',
        r'\usepackage{lmodern}',
        r'\usepackage{amsmath,amssymb}',
        r'\usepackage{array,booktabs,longtable}',
        r'\setlength{\LTleft}{0pt}',
        r'\setlength{\LTright}{0pt}',
        r'\title{' + escape_latex(title) + '}',
        r'\author{' + escape_latex(author) + '}',
        r'\date{}',
        r'\begin{document}',
        r'\maketitle',
        r'\noindent Only the initial layer $k=0$ and the highest nonzero '
        r'layer $k=k_{\max}$ are shown.\\[8pt]',
        ''
    ])


def write_top_bottom_table(lam, mu, filename, standalone=True,
                           max_terms_per_line=2, max_chars=68,
                           title=None, author=''):
    """Write one pair's two-layer LaTeX as a compilable file by default."""
    path = Path(filename)
    path.parent.mkdir(parents=True, exist_ok=True)
    body = table_top_bottom_tex(lam, mu, max_terms_per_line, max_chars)
    if standalone:
        text = (_top_bottom_document_start(title or 'Extremal socle layers', author)
                + body + '\n' + r'\end{document}' + '\n')
    else:
        text = body
    path.write_text(text, encoding='utf-8')
    return path


def generate_top_bottom_tables(max_lam_degree, max_mu_degree, filename,
                               exploit_symmetry=False, standalone=True,
                               max_terms_per_line=2, max_chars=68,
                               title=None, author=''):
    """Write every partition-pair table, retaining only k_max and k=0."""
    nmax, mmax = int(max_lam_degree), int(max_mu_degree)
    if nmax < 0 or mmax < 0:
        raise ValueError('Degree bounds must be nonnegative')
    path = Path(filename)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        if standalone:
            f.write(_top_bottom_document_start(
                title or f'Extremal socle layers through degree {max(nmax,mmax)}',
                author))
        for n in range(nmax+1):
            for m in range(mmax+1):
                if exploit_symmetry and n > m:
                    continue
                for lam in partitions_of(n):
                    for mu in partitions_of(m):
                        if exploit_symmetry and n == m and lam > mu:
                            continue
                        f.write(table_top_bottom_tex(lam, mu,
                            max_terms_per_line, max_chars))
                        f.write('\n')
        if standalone:
            f.write(r'\end{document}' + '\n')
    return path


# Optional aliases matching the older make-table naming style.
MakeTable_top_bottom_tex = write_top_bottom_table
generate_tables_top_bottom = generate_top_bottom_tables


In [17]:

# Preview one LaTeX table, keeping k_max and k=0 only
print(table_top_bottom_tex((2,1,1), (3,1,1)))

# Export a complete LaTeX document for one pair
write_top_bottom_table(
    (2,1,1),
    (3,1,1),
    "tex_tables/top_bottom_example.tex"
)

# Export all partition pairs through degree 9
generate_top_bottom_tables(
    9, 9,
    "tex_tables/top_bottom_deg9.tex",
    exploit_symmetry=True
)


\begingroup
\small
\setlength{\LTpre}{6pt}
\setlength{\LTpost}{6pt}
\begin{longtable}{@{}p{0.14\linewidth}@{\hspace{0.8em}}p{0.79\linewidth}@{}}
\caption{Extremal layers for $\lambda=(2,1,1)$, $\mu=(3,1,1)$}\\
\toprule
Layer & Decomposition \\
\midrule
\endfirsthead
\toprule
Layer & Decomposition (continued) \\
\midrule
\endhead
$k=9$ & $\displaystyle 3\,V^{\varnothing,\varnothing}$ \\
\addlinespace[5pt]\midrule\addlinespace[5pt]
$k=0$ & $\displaystyle V^{(2,1,1),(3,1,1)}$ \\
\bottomrule
\end{longtable}
\endgroup



PosixPath('tex_tables/top_bottom_deg9.tex')